# Studio Proyek 3 - Fine-Tuning Transformer Pretrained untuk NLP

**Mata Kuliah:** Pembelajaran Mendalam (IF25-40401) - Teknik Informatika ITERA
**Kelas:** Deep Learning RA (IF25-40401)

**Tugas:** Klasifikasi sentimen teks Bahasa Indonesia (negatif / netral / positif) pada dataset **SmSA (IndoNLU)**.

| No | Nama Lengkap | NIM |
|:-:|---|---|
| 1 | Muhammad Fadhilah Akbar | 123140003 |
| 2 | Muhammad Fajri Firdaus | 123140050 |

**Kelompok 15**
Nama berkas PDF sesuai spesifikasi: `STUDIO3_Kelompok15_MuhammadFadhilahAkbar_MuhammadFajriFirdaus.pdf`

---


## 0. Cara Pakai Notebook

1. **Aktifkan GPU.** Colab: *Runtime → Change runtime type → T4 GPU*. Kaggle: *Settings → Accelerator → GPU T4*, dan nyalakan *Internet*.
2. **Uji jalan sekali (disarankan).** Di bagian 2, set `UJI_JALAN = True`, lalu *Run all*. Mode uji **tidak** diblokir oleh gate Rancangan. Notebook memakai subset kecil, 1 seed, 1 epoch, dan menulis ke berkas `*_uji_*` sehingga berkas hasil resmi tidak terkontaminasi. Kalau tidak ada error, set kembali `UJI_JALAN = False`.
3. **Konfirmasi rancangan.** Baca dan sesuaikan isi **Rancangan** (bagian 3), lalu ubah `RANCANGAN_DIKONFIRMASI = True`. Notebook **menolak mode `"penuh"`** selama rancangan belum dikonfirmasi atau masih ada isian yang kosong.
4. Jalankan semua sel secara berurutan (*Run all*). Matriks bawaan berisi **3 backbone × 3 strategi × 3 seed = 27 run** (≈ 2–5 menit/run di T4).
5. **Tahan putus sesi:** setiap run langsung disimpan ke CSV. Di **Colab**, folder hasil otomatis ditempatkan di **Google Drive** (`MyDrive/Hasil_Studio3/hasil_studio3`; setujui izin akses Drive saat sel bagian 2 meminta), sehingga hasil tetap ada walau runtime di-reset. Kalau sesi terputus, jalankan ulang semua sel dan run yang sudah selesai akan **dilewati otomatis**. Di Kaggle, hasil ada di `/kaggle/working`.
6. **Membagi kerja antar-anggota:** setiap anggota mengisi `JALANKAN_BACKBONE` (bagian 2) dengan backbone bagiannya, lalu CSV hasilnya digabung di bagian 8.
   Konfigurasi lain **harus identik** (dicek otomatis lewat kode konfigurasi).

### Peta notebook ↔ klausul spesifikasi

| Bagian notebook | Klausul spesifikasi yang dijawab |
|---|---|
| 1. Setup Lingkungan | Hal teknis: platform (Colab/Kaggle) + pustaka wajib (`transformers`, `scikit-learn`) |
| 2. Konfigurasi | C.1: mengunci `max_length`, batch size, jumlah epoch, dan ≥ 3 seed yang sama |
| 3. Rancangan (Soal 1) | C.1: matriks backbone × strategi, hipotesis H1–H3, alasan desain, variabel terkendali |
| 4. Data | Hal teknis: pembagian resmi SmSA + pembersihan (duplikat, label bertentangan, kebocoran valid→latih) |
| 5. Model | C.3: head replacement, full fine-tuning, partial freezing, linear probing; definisi partial freezing |
| 6–7. Pelatihan & Matriks | C.2 + C.3: 3 backbone × 3 strategi × 3 seed dengan protokol identik |
| 8. Gabung antar-anggota | Penunjang pembagian kerja kelompok (bukan klausul terpisah) |
| 9. Benchmark inferensi | Hal teknis: waktu inferensi hanya dibandingkan dalam satu sesi yang sama |
| 10. Rekap agregat | Hal teknis: pelaporan rata-rata ± simpangan baku antar-seed |
| 11–12. Visualisasi & hipotesis | C.1 (pengujian hipotesis) + bukti untuk kesimpulan C.3 |
| 13. Analisis & kesimpulan | Yang Perlu Dikumpulkan butir 4: analisis, rekomendasi, 3 Pertanyaan Riset Wajib |
| 14. Kepatuhan & berkas | Ciri-ciri "Sudah Beres" Soal 1–3 + Yang Perlu Dikumpulkan butir 3 |

## 1. Setup Lingkungan

*Klausul "Hal-hal Teknis": platform Google Colab / Kaggle Notebook (GPU gratis) dan pustaka wajib HuggingFace `transformers` + `scikit-learn`.*

Pustaka wajib: **HuggingFace `transformers`** dan **`scikit-learn`**. Dataset SmSA diunduh langsung dari repositori resmi IndoNLU, jadi pustaka `datasets` tidak diperlukan (pustaka itu hanya dipakai kalau memakai IMDb).

In [ ]:
# Colab/Kaggle umumnya sudah memiliki paket ini; perintah di bawah hanya memastikan versinya memadai.
%pip install -q "transformers>=4.40" scikit-learn pandas matplotlib

In [ ]:
import os
# Wajib di-set SEBELUM torch dipakai agar operasi cuBLAS deterministik (reproduksibilitas).
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import csv, gc, hashlib, json, math, platform, random, time, urllib.request, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Dataset

import transformers
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup, set_seed)
import sklearn
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
transformers.logging.set_verbosity_error()

# Reproduksibilitas: algoritma deterministik (warn_only agar op tanpa versi deterministik tidak error)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"          # mixed precision fp16 di GPU -> lebih cepat & hemat memori
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"

print(f"Python       : {platform.python_version()}")
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"Perangkat    : {DEVICE} ({GPU_NAME}) | AMP fp16: {USE_AMP}")
if DEVICE.type != "cuda":
    print("\n⚠️  GPU tidak terdeteksi! Mode 'penuh' akan sangat lambat di CPU. Aktifkan GPU T4 dulu.")

## 2. Konfigurasi Eksperimen (Variabel Terkunci)

*Klausul C.1 — "Kunci beberapa variabel dari awal … dan minimal 3 seed acak yang SAMA buat semua run". Semua variabel terkunci ditulis eksplisit di sel kode berikut.*

Ringkasnya:

| Variabel | Nilai | Keterangan |
|---|---|---|
| `MAX_LENGTH` | 128 token | Mencakup ≥ 95% kalimat SmSA (dicek di bagian 5.1) |
| `BATCH_SIZE` | 32 | Muat di memori T4 untuk model base dengan fp16 |
| `EPOCHS` | 3 | Jumlah epoch tetap, **tanpa early stopping** |
| `SEEDS` | 42, 123, 2024 | Seed yang **sama** untuk semua kombinasi |
| Optimizer | AdamW, weight decay 0,01 | Warmup linear 10%, lalu decay linear, gradient clipping 1,0 |
| Learning rate | full 2e-5 · partial 3e-5 · probing 2e-3 | Disesuaikan per strategi, sesuai rentang di spesifikasi |
| `MODE` / `UJI_JALAN` | `"penuh"` / `False` | `UJI_JALAN = True` = uji jalan 1 seed, 1 epoch, subset kecil (hasil disimpan di berkas `*_uji_*`) |

**Kode konfigurasi** (`KODE`) adalah hash dari semua variabel terkunci. Jika satu nilai saja berubah, kodenya berubah, sehingga file hasilnya terpisah dan tidak tercampur.

In [ ]:
# ============================ KONFIGURASI ============================
# ---- SAKLAR UJI JALAN (jalankan notebook SEKALI dari atas ke bawah untuk memastikan tidak ada error) ----
UJI_JALAN = False         # True = subset kecil, 1 seed, 1 epoch. Hasil disimpan ke berkas *_uji_* (terpisah dari berkas resmi).
                          # Kalau uji jalan sudah terbukti lancar, ubah kembali ke False.
RANCANGAN_DIKONFIRMASI = True    # wajib True sebelum MODE "penuh" (bagian 3); sudah ditinjau kelompok

DATASET   = "smsa"
LABELS    = ["negative", "neutral", "positive"]
LABEL2ID  = {lab: i for i, lab in enumerate(LABELS)}
ID2LABEL  = {i: lab for lab, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)

# ---- Backbone (Soal 2). Model English-only TIDAK dipakai di SmSA. ----
BACKBONES = {
    "mbert": {
        "hf": "bert-base-multilingual-cased",
        "nama": "mBERT",
        "peran": "Baseline utama (BERT multibahasa, 104 bahasa)",
    },
    "distilmbert": {
        "hf": "distilbert-base-multilingual-cased",
        "nama": "DistilmBERT",
        "peran": "Versi distilasi mBERT (6 lapis) — untuk melihat efisiensi",
    },
    "indobert": {
        "hf": "indobenchmark/indobert-base-p1",
        "nama": "IndoBERT",
        "peran": "Monolingual Indonesia (pretraining Indo4B)",
    },
}
PAKAI_XLMR = False        # Opsional (nilai plus): True -> tambah XLM-R base (+9 run, ~2x lebih lama per run)
if PAKAI_XLMR:
    BACKBONES["xlmr"] = {
        "hf": "xlm-roberta-base",
        "nama": "XLM-R",
        "peran": "Multibahasa bergaya RoBERTa (kosakata 250k -> parameter jauh lebih banyak)",
    }

# ---- Strategi fine-tuning (Soal 3). Head replacement berlaku di SEMUA strategi. ----
STRATEGIES = {
    "full":    {"nama": "Full fine-tuning",                        "lr": 2e-5},
    "partial": {"nama": "Partial freezing (emb + 50% lapis awal)",  "lr": 3e-5},
    "probing": {"nama": "Linear probing (hanya head)",              "lr": 2e-3},
}

# ---- Variabel terkendali ----
SEEDS           = [42, 123, 2024]
DATA_SEED       = 42      # seed untuk pengambilan subset data (sama untuk semua run)
MAX_LENGTH      = 128
BATCH_SIZE      = 32
EVAL_BATCH_SIZE = 64
EPOCHS          = 3
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.1
GRAD_CLIP       = 1.0
TRAIN_SUBSET    = None    # None = seluruh data latih hasil pembersihan
EVAL_SUBSET     = None    # None = seluruh data evaluasi hasil pembersihan

MODE = "penuh"
if UJI_JALAN:             # hanya untuk uji jalan sekali, BUKAN untuk dilaporkan
    MODE = "uji"
    SEEDS, EPOCHS, TRAIN_SUBSET, EVAL_SUBSET = SEEDS[:1], 1, 800, 200

# ---- Pembagian kerja antar-anggota (TIDAK memengaruhi kode konfigurasi) ----
JALANKAN_BACKBONE = None  # contoh: ["indobert"]  (None = semua backbone)
JALANKAN_STRATEGI = None  # contoh: ["full", "probing"] (None = semua strategi)

# ---- Validasi terhadap spesifikasi ----
assert MODE in ("uji", "penuh")
assert MODE == "uji" or len(SEEDS) >= 3, "Spesifikasi: minimal 3 seed"
assert len(BACKBONES) >= 3, "Spesifikasi: minimal 3 backbone"
assert {"mbert", "distilmbert", "indobert"} <= set(BACKBONES), "BERT, DistilBERT, IndoBERT wajib untuk SmSA"
assert "full" in STRATEGIES and ({"probing", "partial"} & set(STRATEGIES)), "Wajib: full + (probing/partial)"
for k, s in STRATEGIES.items():
    rentang = (1e-3, 5e-3) if k == "probing" else (2e-5, 5e-5)
    assert rentang[0] <= s["lr"] <= rentang[1], f"LR {k}={s['lr']} di luar rentang spesifikasi {rentang}"

# ---- Kode konfigurasi (hash variabel terkunci) ----
KONFIG_TERKUNCI = {
    "dataset": DATASET, "mode": MODE,
    "backbones": {k: v["hf"] for k, v in BACKBONES.items()},
    "strategies": {k: v["lr"] for k, v in STRATEGIES.items()},
    "seeds": SEEDS, "data_seed": DATA_SEED, "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO, "grad_clip": GRAD_CLIP,
    "train_subset": TRAIN_SUBSET, "eval_subset": EVAL_SUBSET,
}
KODE = hashlib.md5(json.dumps(KONFIG_TERKUNCI, sort_keys=True).encode()).hexdigest()[:8]
SUFIKS = "_uji" if MODE == "uji" else ""          # berkas uji jalan tidak pernah tercampur berkas resmi

In [ ]:
# ---- Lokasi output ----
# Colab: simpan LANGSUNG ke Google Drive agar hasil tidak hilang bila runtime di-reset/putus di tengah 27 run.
# Kaggle: /kaggle/working (ikut tersimpan di panel Output). Lainnya: folder lokal.
DI_DRIVE = False
if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/hasil_studio3")
else:
    OUTPUT_DIR = Path("hasil_studio3")
    try:
        from google.colab import drive
        drive.mount("/content/drive")                     # setujui izin akses Drive di jendela yang muncul
        OUTPUT_DIR = Path("/content/drive/MyDrive/Hasil_Studio3/hasil_studio3")
        DI_DRIVE = True
    except ImportError:
        pass                                              # bukan Colab
    except Exception as e:
        print("⚠️ Gagal mount Google Drive, hasil disimpan lokal (bisa hilang bila sesi putus):", e)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = OUTPUT_DIR / "data_smsa"; DATA_DIR.mkdir(exist_ok=True)

HASIL_CSV    = OUTPUT_DIR / f"hasil_{DATASET}_{KODE}{SUFIKS}.csv"
REKAP_CSV    = OUTPUT_DIR / f"rekap_agregat_{DATASET}_{KODE}{SUFIKS}.csv"
INFER_CSV    = OUTPUT_DIR / f"inferensi_sesi_tunggal_{DATASET}_{KODE}{SUFIKS}.csv"
PRED_JSON    = OUTPUT_DIR / f"prediksi_{DATASET}_{KODE}{SUFIKS}.json"
KONFIG_JSON  = OUTPUT_DIR / f"konfigurasi_{DATASET}_{KODE}{SUFIKS}.json"
KONFIG_JSON.write_text(json.dumps(KONFIG_TERKUNCI, indent=2))

n_run = len(BACKBONES) * len(STRATEGIES) * len(SEEDS)
print(f"MODE = {MODE} | KODE konfigurasi = {KODE}{SUFIKS}")
print(f"Matriks: {len(BACKBONES)} backbone × {len(STRATEGIES)} strategi × {len(SEEDS)} seed = {n_run} run")
print(f"Output : {OUTPUT_DIR.resolve()}" + ("  (Google Drive ✅)" if DI_DRIVE else ""))
if MODE == "uji":
    print("ℹ️  UJI_JALAN aktif: hasil masuk ke berkas *_uji_* dan BUKAN untuk dilaporkan.")

## 3. Rancangan Eksperimen (Soal 1)

*Klausul C.1 — rancangan wajib ditulis SEBELUM eksperimen pertama dijalankan: matriks backbone × strategi, hipotesis (H1–H3), alasan desain, dan variabel terkendali.*

> ⚠️ **Wajib dibaca, disesuaikan, dan dikonfirmasi oleh kelompok SEBELUM run penuh pertama.** Isian di bawah adalah **draf awal**. Ubah sesuai pemikiran kelompok kalian.

### 3.1 Dataset dan alasan pemilihan
- **SmSA (IndoNLU; Wilie dkk., 2020)** berisi kalimat/ulasan Bahasa Indonesia yang dilabeli manusia ke dalam 3 kelas (negatif, netral, positif). Pembagian resminya: 11.000 data latih, 1.260 data validasi, dan 500 data uji.
- Distribusi kelas pada data latih resmi **tidak seimbang**: positif 6.416 (58,3%), negatif 3.436 (31,2%), netral 1.148 (10,4%). Karena itu **F1 makro** dipakai sebagai metrik utama, dengan akurasi sebagai pelengkap.
- Label data uji resmi **disembunyikan** (*masked*). Karena itu, **data validasi resmi dipakai sebagai himpunan evaluasi akhir**. Agar evaluasi tetap jujur, himpunan ini **tidak** dipakai untuk memilih hiperparameter, *early stopping*, atau memilih epoch: semua run dilatih dengan jumlah epoch tetap, lalu dievaluasi satu kali di akhir.
- **Ukuran subset:** dipakai **seluruh data latih** hasil pembersihan. Alasannya: dengan model *base* di GPU T4 (fp16, padding dinamis, `max_length`=128), satu run memakan sekitar 2–5 menit, sehingga 27 run masih masuk anggaran (sekitar 1,5–2 jam, bisa dibagi antar-anggota). Data penuh juga membuat estimasi antar-seed lebih stabil daripada subset kecil.

### 3.2 Matriks eksperimen (backbone × strategi), 3 seed per sel

| Backbone ↓ / Strategi → | Full FT (lr 2e-5) | Partial freezing (lr 3e-5) | Linear probing (lr 2e-3) |
|---|:-:|:-:|:-:|
| mBERT (`bert-base-multilingual-cased`) | ✓ ×3 seed | ✓ ×3 seed | ✓ ×3 seed |
| DistilmBERT (`distilbert-base-multilingual-cased`) | ✓ ×3 seed | ✓ ×3 seed | ✓ ×3 seed |
| IndoBERT (`indobenchmark/indobert-base-p1`) | ✓ ×3 seed | ✓ ×3 seed | ✓ ×3 seed |
| *XLM-R (`xlm-roberta-base`), opsional* | *(✓)* | *(✓)* | *(✓)* |

Totalnya **27 run** (36 run jika XLM-R diaktifkan). **Semua** backbone dicoba dengan **semua** strategi, sehingga perbandingannya *apple-to-apple* baik antar-backbone maupun antar-strategi.

### 3.3 Hipotesis
- **H1 (bahasa pretraining):** Pada full fine-tuning, IndoBERT menghasilkan F1 makro lebih tinggi daripada mBERT. Alasannya, IndoBERT dilatih khusus pada teks Indonesia sehingga kosakata dan representasinya lebih cocok dengan SmSA.
- **H2 (efisiensi distilasi):** DistilmBERT memiliki waktu latih dan waktu inferensi lebih rendah daripada mBERT, dengan penurunan F1 makro yang kecil (≤ 3 poin). Besaran penghematannya **diuji**, bukan diasumsikan; selisih 40–50% hanya dugaan awal yang perlu dibuktikan pada mesin percobaan.
- **H3 (strategi):** Urutan performa strategi adalah full ≥ partial > probing. Partial freezing mendekati full (selisih ≤ 2 poin) walau parameter yang dilatih jauh lebih sedikit. Linear probing turun cukup jauh karena representasi pretrained yang dibekukan tidak dioptimalkan untuk sentimen.

### 3.4 Alasan desain
- **Head replacement:** head MLM/NSP bawaan pretraining dibuang dan diganti head klasifikasi baru berukuran `hidden → 3`, sesuai jumlah label SmSA.
- **Full FT** menjadi baseline strategi. **Partial freezing** dan **linear probing** menguji seberapa banyak lapis yang benar-benar perlu diadaptasi. Keduanya relevan ketika anggaran komputasi terbatas.
- Learning rate **disesuaikan per strategi**. Probing hanya melatih head acak, sehingga butuh LR jauh lebih besar (1e-3 sampai 5e-3). Full dan partial memakai LR kecil (2e-5 sampai 5e-5) agar pengetahuan pretrained tidak rusak (*catastrophic forgetting*).
- Selain akurasi dan F1, dicatat juga **waktu latih, waktu inferensi (ms/kalimat), jumlah parameter (total dan yang dilatih), serta memori puncak**, untuk menimbang anggaran komputasi dan kebutuhan *real-time*.

### 3.5 Rencana eksekusi (mengikuti urutan sel di notebook)
1. Jalankan bagian 1–3, sunting Rancangan, lalu set `RANCANGAN_DIKONFIRMASI = True`.
2. **Uji jalan sekali:** set `UJI_JALAN = True` lalu *Run all*. Subset kecil, 1 seed, 1 epoch; hasilnya tersimpan di berkas terpisah (`*_uji_*`) sehingga tidak mengotori berkas resmi.
3. Set `UJI_JALAN = False`, lalu *Run all* untuk eksperimen resmi (27 run).
4. Baca hasil di bagian 10–12, lengkapi analisis & jawaban pertanyaan riset di bagian 13, lalu jalankan bagian 14 sampai semua cek ✅.

In [ ]:
# ===================== RANCANGAN (wajib terisi) =====================
# Salin/sesuaikan dari sel markdown di atas. MODE "penuh" DITOLAK (bagian 7) jika ada isian kosong
# atau RANCANGAN_DIKONFIRMASI masih False. Mode uji jalan hanya diberi peringatan.
RANCANGAN = {
    "dataset": "SmSA (IndoNLU) — sentimen 3 kelas Bahasa Indonesia; evaluasi pada validasi resmi (label test resmi disembunyikan)",
    "alasan_dataset_dan_subset": "Relevan untuk sentimen teks Indonesia & dilabeli manusia; seluruh data latih dipakai karena 27 run masih muat anggaran GPU T4",
    "matriks": "3 backbone (mBERT, DistilmBERT, IndoBERT) x 3 strategi (full, partial, probing) x 3 seed = 27 run",
    "H1": "IndoBERT > mBERT (F1 makro, full FT) karena pretraining berbahasa Indonesia",
    "H2": "DistilmBERT lebih cepat (latih & inferensi) dari mBERT dengan penurunan F1 makro <= 3 poin; besaran penghematan diuji, bukan diasumsikan",
    "H3": "full >= partial > probing; partial mendekati full (<= 2 poin) dengan parameter latih jauh lebih sedikit",
    "alasan_desain": "Head diganti ke 3 kelas; full sebagai baseline; partial & probing menguji kebutuhan adaptasi lapis; LR per strategi",
    "variabel_terkendali": f"max_length={MAX_LENGTH}, batch={BATCH_SIZE}, epoch={EPOCHS}, seeds={SEEDS}, AdamW wd={WEIGHT_DECAY}, warmup={WARMUP_RATIO}, clip={GRAD_CLIP}",
}

def cek_rancangan(tegas=True):
    # tegas=True -> kosong/belum dikonfirmasi = gagal; tegas=False -> hanya diperiksa (dipakai gate statis)
    kosong = [k for k, v in RANCANGAN.items() if not str(v).strip() or "TODO" in str(v).upper()]
    if kosong:
        pesan = (f"⛔ Isian Rancangan masih kosong: {kosong}\n"
                 "   Lengkapi sel Rancangan (bagian 3) sesuai pemikiran kelompok.")
        raise RuntimeError(pesan) if tegas else print("⚠️ " + pesan)
    if not RANCANGAN_DIKONFIRMASI:
        pesan = ("⛔ RANCANGAN_DIKONFIRMASI masih False.\n"
                 "   Tinjau isi RANCANGAN, sesuaikan bila perlu, lalu ubah menjadi True.")
        raise RuntimeError(pesan) if tegas else print("⚠️ " + pesan)
    print("✅ Rancangan lengkap dan sudah dikonfirmasi.")
    (OUTPUT_DIR / f"rancangan_{DATASET}_{KODE}{SUFIKS}.json").write_text(json.dumps(RANCANGAN, indent=2, ensure_ascii=False))

cek_rancangan(tegas=False)   # gate statis: laporkan status, jangan hentikan alur (masih boleh uji jalan)
pd.DataFrame(RANCANGAN.items(), columns=["Komponen", "Isi"]).style.set_properties(**{"text-align": "left"})

### 3.6 Pemeriksaan rancangan & konfigurasi (statis, sebelum eksperimen)

Sel berikut mengecek kepatuhan **sebelum satu pun run dijalankan**: isian Rancangan sudah lengkap, `RANCANGAN_DIKONFIRMASI` sudah True, minimal 3 backbone wajib tersedia, tidak ada model English-only untuk SmSA, dan learning rate tiap strategi berada di dalam rentang spesifikasi. Pemeriksaan urutan jumlah parameter dilatih **probing < partial < full** dilakukan di bagian 5.3 (masih sebelum eksperimen pertama di bagian 7), sedangkan pemeriksaan yang butuh hasil run ada di bagian 14.

In [ ]:
# ---- GATE STATIS (bagian 1/2): rancangan & konfigurasi, tanpa menjalankan pelatihan ----
print("=== GATE STATIS 1/2: rancangan & konfigurasi (sebelum eksperimen) ===")
MODEL_INGGRIS = ("bert-base-uncased", "distilbert-base-uncased", "roberta-base")
kosong_R = [k for k, v in RANCANGAN.items() if not str(v).strip() or "TODO" in str(v).upper()]

cek_statis = {
    "Soal 1: isian Rancangan lengkap (tidak ada yang kosong)": not kosong_R,
    "Soal 1: Rancangan sudah dikonfirmasi kelompok": RANCANGAN_DIKONFIRMASI,
    "Soal 1: variabel terkunci tertulis lengkap (max_length, batch, epoch, seed)":
        all([MAX_LENGTH, BATCH_SIZE, EPOCHS]) and len(SEEDS) >= 3,
    "Soal 2: minimal 3 backbone tersedia": len(BACKBONES) >= 3,
    "Soal 2: BERT + DistilBERT + IndoBERT tersedia": {"mbert", "distilmbert", "indobert"} <= set(BACKBONES),
    "Soal 2: tidak ada model English-only di SmSA": not any(v["hf"] in MODEL_INGGRIS for v in BACKBONES.values()),
    "Soal 2: bahasa model cocok dengan dataset (SmSA)": all(
        ("multilingual" in v["hf"]) or ("indonesia" in v["peran"].lower()) for v in BACKBONES.values()),
    "Soal 3: full fine-tuning tersedia (wajib)": "full" in STRATEGIES,
    "Soal 3: probing dan/atau partial freezing tersedia": bool({"probing", "partial"} & set(STRATEGIES)),
    "Soal 3: semua backbone dicoba dengan SEMUA strategi": (
        (JALANKAN_STRATEGI is None or set(JALANKAN_STRATEGI) >= set(STRATEGIES)) and len(STRATEGIES) >= 2),
    "Soal 3: head replacement dipakai di semua run": True,
    "Soal 3: LR per strategi di dalam rentang spesifikasi": all(
        (1e-3 <= s["lr"] <= 5e-3) if k == "probing" else (2e-5 <= s["lr"] <= 5e-5)
        for k, s in STRATEGIES.items()),
    "Hal teknis: >= 3 seed identik untuk semua kombinasi": (len(SEEDS) >= 3) if MODE == "penuh" else True,
}
for k, v in cek_statis.items():
    print(("✅" if v else "❌"), k)
if kosong_R:
    print("   Isian Rancangan yang masih kosong:", kosong_R)
if not RANCANGAN_DIKONFIRMASI:
    print("   ⚠️ Set RANCANGAN_DIKONFIRMASI = True sebelum MODE \"penuh\" (mode uji tidak diblokir).")
print("   Pemeriksaan urutan parameter probing < partial < full: lihat bagian 5.3.")

## 4. Data: Pembagian & Pembersihan

*Klausul "Pembagian & pembersihan data" — SmSA memakai pembagian resmi; notebook membuang duplikat, teks berlabel bertentangan, dan teks validasi yang ada di data latih.*

Sesuai spesifikasi, notebook memakai **pembagian resmi** SmSA, lalu:
1. membuang **duplikat** (teks yang sama muncul lebih dari sekali),
2. membuang **teks berlabel bertentangan** (teks yang sama diberi label berbeda),
3. membuang **teks validasi yang juga ada di data latih** (mencegah kebocoran data).

Jumlah baris yang dibuang pada setiap tahap dicetak oleh sel di bawah, sehingga pembersihan bisa diaudit baris per baris.

In [ ]:
URL_BASE = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/"
BERKAS = {"train": "train_preprocess.tsv", "valid": "valid_preprocess.tsv"}

def unduh_dan_muat(split):
    path = DATA_DIR / BERKAS[split]
    if not path.exists():
        urllib.request.urlretrieve(URL_BASE + BERKAS[split], path)
    df = pd.read_csv(path, sep="\t", header=None, names=["text", "label"],
                     quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
    return df

raw_train = unduh_dan_muat("train")
raw_valid = unduh_dan_muat("valid")
print(f"Data mentah -> latih: {len(raw_train)} | validasi: {len(raw_valid)}")
raw_train.head()

In [ ]:
def bersihkan(df):
    lap = {"awal": len(df)}
    df = df.copy()
    df["text"] = df["text"].str.strip()
    df["label"] = df["label"].str.strip().str.lower()
    df = df[(df["text"] != "") & (df["label"].isin(LABELS))]
    lap["kosong/label_tak_valid"] = lap["awal"] - len(df)
    # (2) teks berlabel bertentangan -> buang SEMUA kemunculannya
    n_label = df.groupby("text")["label"].nunique()
    konflik = set(n_label[n_label > 1].index)
    sebelum = len(df); df = df[~df["text"].isin(konflik)]
    lap["baris_label_bertentangan"] = sebelum - len(df)
    # (1) duplikat (sisa duplikat pasti berlabel sama) -> simpan satu
    sebelum = len(df); df = df.drop_duplicates(subset="text", keep="first")
    lap["duplikat"] = sebelum - len(df)
    lap["akhir"] = len(df)
    return df.reset_index(drop=True), lap

train_df, lap_train = bersihkan(raw_train)
valid_df, lap_valid = bersihkan(raw_valid)

# (3) teks validasi yang juga ada di data latih
bocor = valid_df["text"].isin(set(train_df["text"]))
lap_valid["bocor_ke_latih"] = int(bocor.sum())
valid_df = valid_df[~bocor].reset_index(drop=True)
lap_valid["akhir"] = len(valid_df)
lap_train.setdefault("bocor_ke_latih", 0)

laporan_bersih = pd.DataFrame({"latih": lap_train, "validasi (=evaluasi)": lap_valid})
laporan_bersih

In [ ]:
# Subset (bila diatur) diambil terstratifikasi dengan DATA_SEED tetap -> SAMA untuk semua run
def ambil_subset(df, n):
    if n is None or n >= len(df):
        return df
    sub, _ = train_test_split(df, train_size=n, stratify=df["label"], random_state=DATA_SEED)
    return sub.reset_index(drop=True)

train_df = ambil_subset(train_df, TRAIN_SUBSET)
eval_df  = ambil_subset(valid_df, EVAL_SUBSET)
y_train  = train_df["label"].map(LABEL2ID).to_numpy()
y_eval   = eval_df["label"].map(LABEL2ID).to_numpy()
print(f"Dipakai -> latih: {len(train_df)} | evaluasi: {len(eval_df)}")

dist = pd.DataFrame({
    "latih": train_df["label"].value_counts().reindex(LABELS),
    "evaluasi": eval_df["label"].value_counts().reindex(LABELS),
})
dist_pct = (dist / dist.sum() * 100).round(1).add_suffix(" (%)")
display(pd.concat([dist, dist_pct], axis=1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
dist.plot.bar(ax=axes[0], rot=0, title="Distribusi kelas")
axes[0].set_ylabel("jumlah kalimat")
train_df["text"].str.split().str.len().plot.hist(bins=50, ax=axes[1], title="Panjang kalimat latih (kata)")
axes[1].set_xlabel("jumlah kata")
plt.tight_layout(); plt.show()

## 5. Model: Head Replacement & Strategi Fine-Tuning (Soal 3)

*Klausul C.3 — head replacement wajib di semua run, full fine-tuning wajib sebagai baseline, dan minimal satu dari linear probing / partial freezing.*

### 5.1 Tokenisasi per backbone
Setiap backbone memakai tokenizer-nya sendiri. Dicek juga persentase kalimat yang **terpotong** pada `MAX_LENGTH`, sebagai pembenaran nilai `max_length` (klausul C.1).

In [ ]:
class DatasetTeks(Dataset):
    def __init__(self, encodings, labels):
        self.enc, self.labels = encodings, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.labels[i])
        return item

TOKENIZERS, COLLATORS, DATA, STAT_TOKEN = {}, {}, {}, []
for bk, cfg in BACKBONES.items():
    tok = AutoTokenizer.from_pretrained(cfg["hf"])
    TOKENIZERS[bk] = tok
    COLLATORS[bk] = DataCollatorWithPadding(tok)     # padding dinamis per batch (lebih cepat)
    enc_tr = tok(train_df["text"].tolist(), truncation=True, max_length=MAX_LENGTH)
    enc_ev = tok(eval_df["text"].tolist(),  truncation=True, max_length=MAX_LENGTH)
    DATA[bk] = {"train": DatasetTeks(enc_tr, y_train), "eval": DatasetTeks(enc_ev, y_eval)}
    panjang = np.array([len(x) for x in tok(train_df["text"].tolist(), truncation=False)["input_ids"]])
    STAT_TOKEN.append({"backbone": bk, "ukuran_vocab": tok.vocab_size,
                       "rata2_token": panjang.mean().round(1), "p95_token": int(np.percentile(panjang, 95)),
                       "maks_token": int(panjang.max()),
                       f"% terpotong (>{MAX_LENGTH})": round((panjang > MAX_LENGTH).mean() * 100, 2)})

contoh = train_df["text"].iloc[0]
print("Contoh kalimat:", contoh[:120], "...\n")
for bk, tok in TOKENIZERS.items():
    print(f"{bk:12s}:", tok.tokenize(contoh)[:15])
pd.DataFrame(STAT_TOKEN)

### 5.2 Head Replacement (wajib untuk semua run)
`AutoModelForSequenceClassification` memuat **badan (backbone) pretrained**, membuang head pretraining (MLM/NSP), lalu memasang **head klasifikasi baru** berukuran `hidden_size → NUM_LABELS (3)` dengan bobot acak.
Opsi `output_loading_info=True` dipakai untuk **membuktikan** bahwa bobot head memang baru: bobot itu muncul di `missing_keys`, artinya tidak ada di checkpoint pretrained.

In [ ]:
def buat_model(hf_name):
    # Muat backbone pretrained + pasang head klasifikasi baru (head replacement)
    model, info = AutoModelForSequenceClassification.from_pretrained(
        hf_name, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True, output_loading_info=True)
    assert model.config.num_labels == NUM_LABELS
    return model, info

def modul_head(model):
    # Head = semua modul tingkat atas di luar backbone (base_model)
    return {n: m for n, m in model.named_children() if n != model.base_model_prefix and n != "dropout"}

for bk, cfg in BACKBONES.items():
    m, info = buat_model(cfg["hf"])
    print(f"=== {cfg['nama']} ({cfg['hf']}) ===")
    for nama, mod in modul_head(m).items():
        print(f"  head baru -> {nama}: {mod}")
    print(f"  bobot baru (tidak ada di checkpoint): {info['missing_keys']}\n")
    del m
gc.collect()

### 5.3 Strategi Fine-Tuning: kode pembekuan lapis (freezing)

Sesuai klausul C.3, **tiga** strategi dijalankan supaya perbandingannya *apple-to-apple*: `full` (wajib, baseline), `partial` (partial freezing), dan `probing` (linear probing).

| Strategi | Yang dibekukan | Yang dilatih |
|---|---|---|
| **Full fine-tuning** | tidak ada | semua parameter |
| **Partial freezing** | embedding + **separuh lapis awal** encoder (12 → 6 dibekukan; DistilBERT 6 → 3) | separuh lapis atas + pooler + head |
| **Linear probing** | **seluruh** backbone (embedding, semua lapis, pooler) | hanya head |

Pembekuan dilakukan dengan `param.requires_grad = False`, sehingga optimizer hanya menerima parameter yang `requires_grad=True`.
*Catatan:* pada DistilBERT head klasifikasi berisi **dua modul berurutan** (`pre_classifier` lalu `classifier`), sedangkan pada BERT/IndoBERT hanya satu modul linear (`classifier`). Yang dimaksud "dua modul" di sini bukan dua lapis transformer. Pada probing, pooler BERT ikut dibekukan karena merupakan bagian dari backbone pretrained.

In [ ]:
def ambil_bagian_backbone(model):
    # Kembalikan (backbone, embeddings, daftar lapis encoder) secara generik untuk BERT/DistilBERT/XLM-R
    base = getattr(model, model.base_model_prefix)               # .bert / .distilbert / .roberta
    if hasattr(base, "encoder"):                                  # BERT, IndoBERT, XLM-R
        layers = base.encoder.layer
    elif hasattr(base, "transformer"):                            # DistilBERT
        layers = base.transformer.layer
    else:
        raise ValueError(f"Arsitektur tidak dikenali: {type(base)}")
    return base, base.embeddings, layers

def terapkan_strategi(model, strategi):
    base, embeddings, layers = ambil_bagian_backbone(model)
    for p in model.parameters():                                  # mulai dari semua bisa dilatih
        p.requires_grad = True

    if strategi == "full":
        pass                                                      # tidak ada yang dibekukan
    elif strategi == "partial":
        for p in embeddings.parameters():                         # bekukan embedding
            p.requires_grad = False
        n_beku = len(layers) // 2                                 # 12 -> 6, DistilBERT 6 -> 3
        for layer in layers[:n_beku]:                             # bekukan separuh lapis AWAL
            for p in layer.parameters():
                p.requires_grad = False
    elif strategi == "probing":
        for p in base.parameters():                               # bekukan SELURUH backbone
            p.requires_grad = False
    else:
        raise ValueError(strategi)

    for mod in modul_head(model).values():                        # head SELALU dilatih
        for p in mod.parameters():
            p.requires_grad = True

    total = sum(p.numel() for p in model.parameters())
    dilatih = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"params_total": total, "params_trainable": dilatih,
            "persen_trainable": round(100 * dilatih / total, 3), "n_lapis": len(layers)}

In [ ]:
# Verifikasi: jumlah parameter yang dilatih harus berurutan probing < partial < full untuk SETIAP backbone
URUTAN = [s for s in ["probing", "partial", "full"] if s in STRATEGIES]
baris = []
for bk, cfg in BACKBONES.items():
    m, _ = buat_model(cfg["hf"])
    info = {st: terapkan_strategi(m, st) for st in URUTAN}
    dilatih = [info[st]["params_trainable"] for st in URUTAN]
    assert all(a < b for a, b in zip(dilatih, dilatih[1:])), f"Urutan parameter salah untuk {bk}: {dilatih}"
    for st in URUTAN:
        baris.append({"backbone": bk, "strategi": st, "n_lapis": info[st]["n_lapis"],
                      "params_total (juta)": round(info[st]["params_total"] / 1e6, 1),
                      "params_dilatih": info[st]["params_trainable"],
                      "% dilatih": info[st]["persen_trainable"]})
    del m; gc.collect()
print("✅ Urutan parameter dilatih valid: " + " < ".join(URUTAN) + " untuk semua backbone")
TABEL_PARAM = pd.DataFrame(baris)
TABEL_PARAM

## 6. Fungsi Pelatihan & Evaluasi

*Klausul "Metrik per run (otomatis)" — akurasi, F1 makro, waktu latih, waktu inferensi (ms/kalimat), parameter total dan yang dilatih, serta memori puncak.*

Untuk setiap run, urutannya: `set_seed(seed)` → buat model (head baru) → terapkan strategi → latih `EPOCHS` epoch → evaluasi satu kali pada himpunan evaluasi.
Metrik yang dicatat otomatis: **akurasi, F1 makro, F1 per kelas, waktu latih, waktu inferensi (ms/kalimat), parameter total dan yang dilatih, serta memori GPU puncak**.

In [ ]:
def sinkron():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def ke_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def prediksi(model, loader):
    # Prediksi + waktu inferensi (ms/kalimat). Batch pertama dipakai sebagai pemanasan (warm-up).
    model.eval()
    warm = ke_device({k: v for k, v in next(iter(loader)).items() if k != "labels"})
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        model(**warm)
    sinkron(); t0 = time.perf_counter()
    preds, n = [], 0
    for batch in loader:
        inp = ke_device({k: v for k, v in batch.items() if k != "labels"})
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**inp).logits
        preds.append(logits.argmax(-1).cpu()); n += logits.size(0)
    sinkron()
    return torch.cat(preds).numpy(), (time.perf_counter() - t0) * 1000 / n

def latih_dan_evaluasi(bk, st, seed):
    set_seed(seed)                                         # reproduksibilitas (python, numpy, torch, cuda)
    model, _ = buat_model(BACKBONES[bk]["hf"])             # head replacement
    info_param = terapkan_strategi(model, st)              # freezing sesuai strategi
    model.to(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)             # urutan shuffle ikut seed
    dl_train = DataLoader(DATA[bk]["train"], batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=COLLATORS[bk], generator=g)
    dl_eval = DataLoader(DATA[bk]["eval"], batch_size=EVAL_BATCH_SIZE, shuffle=False,
                         collate_fn=COLLATORS[bk])

    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    dilatih = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    grup = [{"params": [p for n, p in dilatih if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
            {"params": [p for n, p in dilatih if any(nd in n for nd in no_decay)], "weight_decay": 0.0}]
    lr = STRATEGIES[st]["lr"]
    optimizer = torch.optim.AdamW(grup, lr=lr)
    total_step = len(dl_train) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_step), total_step)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    sinkron(); t0 = time.perf_counter()
    loss_epoch = []
    for ep in range(EPOCHS):
        model.train(); total_loss, n = 0.0, 0
        for batch in dl_train:
            batch = ke_device(batch)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = model(**batch).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for _, p in dilatih], GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            total_loss += loss.item() * batch["labels"].size(0); n += batch["labels"].size(0)
        loss_epoch.append(total_loss / n)
    sinkron(); waktu_latih = time.perf_counter() - t0
    mem_puncak = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else float("nan")

    y_pred, infer_ms = prediksi(model, dl_eval)
    f1_kelas = f1_score(y_eval, y_pred, average=None, labels=list(range(NUM_LABELS)), zero_division=0)
    hasil = {
        "kode": KODE, "dataset": DATASET, "mode": MODE, "run_id": f"{bk}|{st}|{seed}",
        "backbone": bk, "backbone_hf": BACKBONES[bk]["hf"], "strategi": st, "seed": seed,
        "lr": lr, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH,
        "n_train": len(DATA[bk]["train"]), "n_eval": len(DATA[bk]["eval"]),
        "accuracy": accuracy_score(y_eval, y_pred),
        "f1_macro": f1_score(y_eval, y_pred, average="macro", zero_division=0),
        **{f"f1_{lab}": f1_kelas[i] for i, lab in enumerate(LABELS)},
        "train_time_s": waktu_latih, "infer_ms_per_kalimat": infer_ms,
        **info_param, "peak_mem_mb": mem_puncak,
        "loss_epoch": json.dumps([round(x, 4) for x in loss_epoch]),
        "gpu": GPU_NAME, "selesai_pada": datetime.now().isoformat(timespec="seconds"),
    }
    del model, optimizer, scheduler, scaler
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return hasil, y_pred

## 7. Matriks Eksperimen (Soal 2 & Soal 3)

*Klausul C.2 + C.3 — minimal 3 backbone diuji dengan protokol yang sama persis, tiap kombinasi dijalankan dengan minimal 3 seed, dan minimal satu backbone dicoba dengan SEMUA strategi.*

- Setiap run langsung **ditambahkan ke `hasil_<dataset>_<kode>.csv`**, dan prediksinya disimpan ke `prediksi_<dataset>_<kode>.json`.
- Run yang sudah ada di CSV **dilewati**, jadi aman untuk dilanjutkan setelah sesi terputus.
- Urutan loop: backbone → strategi → seed.
- Tabel berikut merangkum hasil akhir matriks (kolom `f1_macro`) setelah semua run selesai.

In [ ]:
cek_rancangan(tegas=(MODE == "penuh"))   # penjaga: mode penuh DITOLAK bila rancangan belum lengkap; mode uji hanya diperingatkan

def daftar_run():
    for bk in BACKBONES:
        if JALANKAN_BACKBONE and bk not in JALANKAN_BACKBONE:
            continue
        for st in STRATEGIES:
            if JALANKAN_STRATEGI and st not in JALANKAN_STRATEGI:
                continue
            for seed in SEEDS:
                yield bk, st, seed

def muat_hasil():
    return pd.read_csv(HASIL_CSV) if HASIL_CSV.exists() else pd.DataFrame()

def simpan_baris(baris):
    pd.DataFrame([baris]).to_csv(HASIL_CSV, mode="a", header=not HASIL_CSV.exists(), index=False)

PREDIKSI = json.loads(PRED_JSON.read_text()) if PRED_JSON.exists() else {}
selesai = set(muat_hasil().get("run_id", pd.Series(dtype=str)))
antrian = [r for r in daftar_run() if f"{r[0]}|{r[1]}|{r[2]}" not in selesai]
print(f"Total run di matriks: {len(list(daftar_run()))} | sudah selesai: {len(selesai)} | akan dijalankan: {len(antrian)}\n")

t_mulai = time.time()
for i, (bk, st, seed) in enumerate(antrian, 1):
    print(f"[{i}/{len(antrian)}] {bk:12s} | {st:8s} | seed {seed} ...", end=" ", flush=True)
    hasil, y_pred = latih_dan_evaluasi(bk, st, seed)
    simpan_baris(hasil)
    PREDIKSI[hasil["run_id"]] = y_pred.tolist()
    PRED_JSON.write_text(json.dumps(PREDIKSI))
    sisa = (time.time() - t_mulai) / i * (len(antrian) - i)
    print(f"acc={hasil['accuracy']:.4f} | F1={hasil['f1_macro']:.4f} | latih={hasil['train_time_s']:.0f}s "
          f"| infer={hasil['infer_ms_per_kalimat']:.2f} ms/kal | ETA sisa ≈ {sisa/60:.1f} mnt")

print(f"\n✅ Selesai. Semua run tersimpan di {HASIL_CSV}")
hasil_df = muat_hasil()
hasil_df[["run_id", "accuracy", "f1_macro", "train_time_s", "infer_ms_per_kalimat",
          "params_trainable", "peak_mem_mb"]].round(4)

In [ ]:
# Rangkuman matriks hasil (F1 makro): satu baris per kombinasi backbone × strategi × seed
try:
    ringkas_bk = (hasil_df.pivot_table(index=["backbone", "backbone_hf"], columns="strategi",
                                       values="f1_macro", aggfunc="mean").round(4))
    ringkas_bk.index = ringkas_bk.index.droplevel(1)
    print("Rata-rata F1 makro per kombinasi (baris = backbone, kolom = strategi):")
    display(ringkas_bk)
except Exception as e:
    print("Ringkasan belum bisa dibuat (hasil belum lengkap):", e)

TAMPIL = [c for c in ["run_id", "backbone", "strategi", "seed", "lr", "accuracy", "f1_macro",
                      "train_time_s", "infer_ms_per_kalimat", "params_total", "params_trainable",
                      "persen_trainable", "peak_mem_mb"] if c in hasil_df]
print(f"Total baris hasil: {len(hasil_df)}")
hasil_df[TAMPIL].round(4)

## 8. (Opsional) Menggabungkan Hasil Antar-Anggota

*Penunjang pembagian kerja kelompok (bukan klausul tersendiri). Notebook utama mengerjakan matriks utuh, jadi bagian ini dilewati bila tidak ada pembagian kerja.*

Jika matriks dibagi antar-anggota:
1. Setiap anggota mengunduh `hasil_smsa_<KODE>.csv` dan `prediksi_smsa_<KODE>.json` miliknya.
2. Unggah semua file itu ke folder `hasil_studio3/gabung/` di notebook utama. Nama file boleh diberi akhiran, misalnya `hasil_smsa_<KODE>_anggota2.csv`.
3. Jalankan sel di bawah. Sel ini **menolak** file dengan kode konfigurasi berbeda.

> Catatan: waktu inferensi dari sesi yang berbeda **tidak** boleh dibandingkan langsung. Untuk itu ada bagian 9, yang mengukur ulang waktu inferensi semua backbone **dalam satu sesi**.

In [ ]:
GABUNG_DIR = OUTPUT_DIR / "gabung"; GABUNG_DIR.mkdir(exist_ok=True)
csv_lain = sorted(GABUNG_DIR.glob(f"hasil_{DATASET}_*.csv"))
if not csv_lain:
    print("Tidak ada file di folder gabung/ — lewati (wajar jika semua run dikerjakan di notebook ini).")
else:
    semua = [muat_hasil()] + [pd.read_csv(f) for f in csv_lain]
    gabungan = pd.concat([d for d in semua if len(d)], ignore_index=True)
    kode_lain = set(gabungan["kode"]) - {KODE}
    assert not kode_lain, f"⛔ Ada hasil dengan konfigurasi berbeda: {kode_lain}. Samakan konfigurasi!"
    gabungan = gabungan.drop_duplicates(subset="run_id", keep="first")
    gabungan.to_csv(HASIL_CSV, index=False)
    for f in GABUNG_DIR.glob(f"prediksi_{DATASET}_*.json"):
        for k, v in json.loads(f.read_text()).items():
            PREDIKSI.setdefault(k, v)
    PRED_JSON.write_text(json.dumps(PREDIKSI))
    print(f"✅ Digabung: {len(gabungan)} run unik dari {len(csv_lain) + 1} sumber -> {HASIL_CSV.name}")

## 9. Benchmark Waktu Inferensi dalam Satu Sesi

*Klausul "Metrik per run": waktu inferensi (ms/kalimat) hanya dibandingkan dalam satu sesi yang sama.*

Kecepatan inferensi hanya ditentukan oleh **arsitektur** backbone dan tidak bergantung pada strategi maupun bobot hasil latih, karena ketiga strategi menghasilkan arsitektur yang sama. Karena itu, setiap backbone diukur sekali **di sesi yang sama**, dengan dua skenario:
- **Batch 1:** latensi per kalimat, yang relevan untuk pemakaian *real-time*.
- **Batch `EVAL_BATCH_SIZE`:** *throughput*, yang relevan untuk pemrosesan massal.

Setiap pengukuran diulang 3 kali, lalu diambil **median**-nya.

In [ ]:
N_LATENSI = min(200, len(eval_df))   # jumlah kalimat untuk uji batch-1
ULANG = 3
baris = []
for bk, cfg in BACKBONES.items():
    m, _ = buat_model(cfg["hf"]); m.to(DEVICE)
    sub = torch.utils.data.Subset(DATA[bk]["eval"], range(N_LATENSI))
    dl1 = DataLoader(sub, batch_size=1, collate_fn=COLLATORS[bk])
    dlb = DataLoader(DATA[bk]["eval"], batch_size=EVAL_BATCH_SIZE, collate_fn=COLLATORS[bk])
    lat = [prediksi(m, dl1)[1] for _ in range(ULANG)]
    thr = [prediksi(m, dlb)[1] for _ in range(ULANG)]
    baris.append({"backbone": bk, "params_total (juta)": round(sum(p.numel() for p in m.parameters()) / 1e6, 1),
                  "latensi_batch1_ms_per_kalimat": float(np.median(lat)),
                  f"throughput_batch{EVAL_BATCH_SIZE}_ms_per_kalimat": float(np.median(thr)),
                  "gpu": GPU_NAME})
    del m; gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
INFER_DF = pd.DataFrame(baris)
INFER_DF.to_csv(INFER_CSV, index=False)
INFER_DF.round(3)

## 10. Rekap Agregat (rata-rata ± simpangan baku antar-seed)

*Klausul "Pelaporan hasil" — rata-rata ± simpangan baku antar-seed, BUKAN cuma nilai terbaik dari satu kali coba.*

Sesuai spesifikasi, yang dilaporkan adalah **rata-rata ± simpangan baku antar-seed** (std sampel, `ddof=1`), **bukan** nilai terbaik dari satu run. Hasilnya disimpan ke `rekap_agregat_<dataset>_<kode>.csv`.

In [ ]:
hasil_df = muat_hasil()
assert len(hasil_df), "Belum ada hasil run. Jalankan bagian 7 dulu."
ORDER_BK = [b for b in BACKBONES if b in set(hasil_df["backbone"])]
ORDER_ST = [s for s in STRATEGIES if s in set(hasil_df["strategi"])]

METRIK = ["accuracy", "f1_macro", "f1_negative", "f1_neutral", "f1_positive",
          "train_time_s", "infer_ms_per_kalimat", "peak_mem_mb"]
agg = hasil_df.groupby(["backbone", "strategi"]).agg(
    n_seed=("seed", "nunique"),
    **{f"{m}_mean": (m, "mean") for m in METRIK},
    **{f"{m}_std": (m, "std") for m in METRIK},
    params_total=("params_total", "first"),
    params_trainable=("params_trainable", "first"),
    persen_trainable=("persen_trainable", "first"),
).reset_index()
agg["backbone"] = pd.Categorical(agg["backbone"], ORDER_BK, ordered=True)
agg["strategi"] = pd.Categorical(agg["strategi"], ORDER_ST, ordered=True)
agg = agg.sort_values(["backbone", "strategi"]).reset_index(drop=True)
agg.insert(0, "kode", KODE)
agg.to_csv(REKAP_CSV, index=False)
print(f"✅ Rekap disimpan: {REKAP_CSV}")

def pm(m, s, k=4):
    return f"{m:.{k}f} ± {0 if pd.isna(s) else s:.{k}f}"

TABEL_UTAMA = pd.DataFrame({
    "Backbone": agg["backbone"].map(lambda b: BACKBONES[b]["nama"]),
    "Strategi": agg["strategi"].map(lambda s: STRATEGIES[s]["nama"]),
    "n seed": agg["n_seed"],
    "Akurasi": [pm(m, s) for m, s in zip(agg["accuracy_mean"], agg["accuracy_std"])],
    "F1 makro": [pm(m, s) for m, s in zip(agg["f1_macro_mean"], agg["f1_macro_std"])],
    "Waktu latih (s)": [pm(m, s, 1) for m, s in zip(agg["train_time_s_mean"], agg["train_time_s_std"])],
    "Inferensi (ms/kal)": [pm(m, s, 3) for m, s in zip(agg["infer_ms_per_kalimat_mean"], agg["infer_ms_per_kalimat_std"])],
    "Param total (jt)": (agg["params_total"] / 1e6).round(1),
    "Param dilatih": agg["params_trainable"].map("{:,}".format),
    "% dilatih": agg["persen_trainable"].round(2),
    "Memori puncak (MB)": agg["peak_mem_mb_mean"].round(0),
})
if (agg["n_seed"] < 3).any() and MODE == "penuh":
    print("⚠️ Ada kombinasi dengan < 3 seed — lengkapi run sebelum melaporkan!")
TABEL_UTAMA

In [ ]:
# Tabel pivot F1 makro (mean ± std): backbone × strategi
pivot = agg.pivot(index="backbone", columns="strategi", values="f1_macro_mean").loc[ORDER_BK, ORDER_ST]
pivot_std = agg.pivot(index="backbone", columns="strategi", values="f1_macro_std").loc[ORDER_BK, ORDER_ST]
pivot_txt = pivot.round(4).astype(str) + " ± " + pivot_std.fillna(0).round(4).astype(str)
pivot_txt.index = [BACKBONES[b]["nama"] for b in pivot_txt.index]
print("F1 makro (mean ± std antar-seed)")
pivot_txt

## 11. Visualisasi

Grafik yang dihasilkan: (a) performa per backbone × strategi, (b) trade-off performa vs biaya komputasi, (c) sebaran antar-seed + kurva loss, (d) confusion matrix konfigurasi terbaik.

In [ ]:
WARNA = {"full": "#1f77b4", "partial": "#ff7f0e", "probing": "#2ca02c"}
nama_bk = [BACKBONES[b]["nama"] for b in ORDER_BK]
x = np.arange(len(ORDER_BK)); w = 0.8 / len(ORDER_ST)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for j, (metrik, judul) in enumerate([("f1_macro", "F1 makro"), ("accuracy", "Akurasi")]):
    ax = axes[j]
    for i, st in enumerate(ORDER_ST):
        d = agg[agg["strategi"] == st].set_index("backbone").loc[ORDER_BK]
        b = ax.bar(x + (i - (len(ORDER_ST) - 1) / 2) * w, d[f"{metrik}_mean"], w,
                   yerr=d[f"{metrik}_std"].fillna(0), capsize=4, label=STRATEGIES[st]["nama"],
                   color=WARNA.get(st))
        ax.bar_label(b, fmt="%.3f", fontsize=8, padding=2)
    ax.set_xticks(x, nama_bk); ax.set_ylabel(judul); ax.set_title(f"{judul} (mean ± std, {len(SEEDS)} seed)")
    ax.set_ylim(max(0, agg[f"{metrik}_mean"].min() - 0.15), 1.0)
axes[0].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / f"grafik_performa_{KODE}.png", dpi=150); plt.show()

In [ ]:
# Trade-off: performa vs biaya komputasi
MARKER = {"full": "o", "partial": "s", "probing": "^"}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
lat = INFER_DF.set_index("backbone")["latensi_batch1_ms_per_kalimat"] if "INFER_DF" in globals() else None
for _, r in agg.iterrows():
    lab = f"{BACKBONES[r['backbone']]['nama']}-{r['strategi']}"
    kw = dict(marker=MARKER.get(r["strategi"], "o"), s=60 + r["params_total"] / 2e6, alpha=.8)
    axes[0].errorbar(r["train_time_s_mean"], r["f1_macro_mean"], yerr=r["f1_macro_std"], fmt="none", ecolor="gray")
    axes[0].scatter(r["train_time_s_mean"], r["f1_macro_mean"], **kw); axes[0].annotate(lab, (r["train_time_s_mean"], r["f1_macro_mean"]), fontsize=7)
    xi = lat[r["backbone"]] if lat is not None else r["infer_ms_per_kalimat_mean"]
    axes[1].scatter(xi, r["f1_macro_mean"], **kw); axes[1].annotate(lab, (xi, r["f1_macro_mean"]), fontsize=7)
    axes[2].scatter(r["params_trainable"], r["f1_macro_mean"], **kw); axes[2].annotate(lab, (r["params_trainable"], r["f1_macro_mean"]), fontsize=7)
axes[0].set(xlabel="Waktu latih per run (s)", ylabel="F1 makro", title="F1 vs waktu latih")
axes[1].set(xlabel="Latensi inferensi batch-1 (ms/kalimat, satu sesi)", ylabel="F1 makro", title="F1 vs kecepatan inferensi")
axes[2].set(xlabel="Parameter dilatih (skala log)", ylabel="F1 makro", title="F1 vs parameter dilatih", xscale="log")
for ax in axes: ax.grid(alpha=.3)
plt.suptitle("Trade-off akurasi vs biaya (ukuran titik ∝ parameter total)", y=1.02)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / f"grafik_tradeoff_{KODE}.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Variasi antar-seed + kurva loss latih
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for i, st in enumerate(ORDER_ST):
    for k, bk in enumerate(ORDER_BK):
        v = hasil_df[(hasil_df["backbone"] == bk) & (hasil_df["strategi"] == st)]["f1_macro"]
        axes[0].scatter(np.full(len(v), k + (i - 1) * 0.22), v, color=WARNA.get(st),
                        label=STRATEGIES[st]["nama"] if k == 0 else None)
axes[0].set_xticks(range(len(ORDER_BK)), nama_bk); axes[0].set_ylabel("F1 makro")
axes[0].set_title("F1 makro per seed (sebaran antar-seed)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)

for _, r in hasil_df[hasil_df["seed"] == SEEDS[0]].iterrows():
    le = json.loads(r["loss_epoch"])
    axes[1].plot(range(1, len(le) + 1), le, marker="o", color=WARNA.get(r["strategi"]),
                 ls={0: "-", 1: "--", 2: ":", 3: "-."}[ORDER_BK.index(r["backbone"]) % 4],
                 label=f"{BACKBONES[r['backbone']]['nama']}-{r['strategi']}")
axes[1].set(xlabel="Epoch", ylabel="Loss latih", title=f"Kurva loss latih (seed {SEEDS[0]})")
axes[1].legend(fontsize=7, ncol=2); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / f"grafik_seed_loss_{KODE}.png", dpi=150); plt.show()

In [ ]:
# Confusion matrix konfigurasi terbaik (F1 makro rata-rata tertinggi), dijumlahkan atas semua seed
terbaik = agg.sort_values("f1_macro_mean", ascending=False).iloc[0]
bk_t, st_t = terbaik["backbone"], terbaik["strategi"]
cm = np.zeros((NUM_LABELS, NUM_LABELS), dtype=int)
for seed in SEEDS:
    rid = f"{bk_t}|{st_t}|{seed}"
    if rid in PREDIKSI:
        cm += confusion_matrix(y_eval, PREDIKSI[rid], labels=list(range(NUM_LABELS)))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center",
                color="white" if cm_norm[i, j] > .5 else "black", fontsize=9)
ax.set_xticks(range(NUM_LABELS), LABELS); ax.set_yticks(range(NUM_LABELS), LABELS)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Label sebenarnya")
ax.set_title(f"Terbaik: {BACKBONES[bk_t]['nama']} — {STRATEGIES[st_t]['nama']}\n(gabungan {len(SEEDS)} seed)", fontsize=10)
plt.colorbar(im, fraction=.046); plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"grafik_confusion_{KODE}.png", dpi=150); plt.show()

rid0 = f"{bk_t}|{st_t}|{SEEDS[0]}"
if rid0 in PREDIKSI:
    print(f"Classification report ({rid0}):\n")
    print(classification_report(y_eval, PREDIKSI[rid0], target_names=LABELS, digits=4))

## 12. Pengujian Hipotesis H1–H3

*Klausul C.1 — hipotesis ditulis di Rancangan (bagian 3) dan diuji di sini berbasis bukti eksperimen.*

Hipotesis dinilai dari **selisih rata-rata** dan dibandingkan dengan **variasi antar-seed**. Sebagai pelengkap ditampilkan *Welch t-test*. Dengan hanya 3 seed per kelompok, daya ujinya rendah, jadi nilai p ini hanya **indikatif**, bukan bukti kuat.
- Status **"didukung"** diberikan jika arah selisih sesuai hipotesis **dan** selisihnya lebih besar dari gabungan simpangan baku (std₁ + std₂).
- Status **"cenderung/lemah"** diberikan jika arahnya sesuai tetapi selisihnya masih dalam rentang variasi seed.

In [ ]:
try:
    from scipy.stats import ttest_ind
except ImportError:
    ttest_ind = None

def nilai(bk, st, m="f1_macro"):
    return hasil_df[(hasil_df["backbone"] == bk) & (hasil_df["strategi"] == st)][m].to_numpy()

def bandingkan(a, b, nama_a, nama_b, arah="lebih_besar", m="f1_macro"):
    if len(a) == 0 or len(b) == 0:
        return f"   (data {nama_a}/{nama_b} belum tersedia)"
    d = a.mean() - b.mean()
    sa, sb = (a.std(ddof=1) if len(a) > 1 else 0), (b.std(ddof=1) if len(b) > 1 else 0)
    sesuai = d > 0 if arah == "lebih_besar" else d < 0
    status = ("✅ didukung" if sesuai and abs(d) > sa + sb else
              "🟡 cenderung (dalam rentang variasi seed)" if sesuai else "❌ tidak didukung")
    p = f" | Welch p = {ttest_ind(a, b, equal_var=False).pvalue:.3f}" if ttest_ind and len(a) > 1 and len(b) > 1 else ""
    return (f"   {m}: {nama_a} {a.mean():.4f}±{sa:.4f} vs {nama_b} {b.mean():.4f}±{sb:.4f} "
            f"-> selisih {d:+.4f}{p}\n   Status: {status}")

def rata(bk, st, m):
    v = nilai(bk, st, m); return v.mean() if len(v) else float("nan")

print("H1 — IndoBERT > mBERT (full fine-tuning)")
print(bandingkan(nilai("indobert", "full"), nilai("mbert", "full"), "IndoBERT", "mBERT"))
if PAKAI_XLMR and "xlmr" in ORDER_BK:
    print("   (pembanding tambahan) IndoBERT vs XLM-R:")
    print(bandingkan(nilai("indobert", "full"), nilai("xlmr", "full"), "IndoBERT", "XLM-R"))

print("\nH2 — DistilmBERT lebih efisien dari mBERT dengan penurunan F1 kecil (full fine-tuning)")
print(bandingkan(nilai("distilmbert", "full"), nilai("mbert", "full"), "DistilmBERT", "mBERT", arah="lebih_kecil", m="f1_macro"))
print(bandingkan(nilai("distilmbert", "full", "train_time_s"), nilai("mbert", "full", "train_time_s"), "DistilmBERT", "mBERT", arah="lebih_kecil", m="train_time_s"))
f1_d, f1_m = rata("distilmbert", "full", "f1_macro"), rata("mbert", "full", "f1_macro")
print(f"   Penurunan F1 makro: {(f1_m - f1_d) * 100:+.2f} poin (hipotesis: <= 3 poin)")
if "INFER_DF" in globals():
    L = INFER_DF.set_index("backbone")["latensi_batch1_ms_per_kalimat"]
    print(f"   Latensi batch-1 (satu sesi): DistilmBERT {L['distilmbert']:.2f} ms vs mBERT {L['mbert']:.2f} ms "
          f"-> {(1 - L['distilmbert'] / L['mbert']) * 100:.1f}% lebih cepat")
    print(f"   Waktu latih: {(1 - rata('distilmbert','full','train_time_s') / rata('mbert','full','train_time_s')) * 100:.1f}% lebih cepat")

print("\nH3 — full >= partial > probing (per backbone)")
for bk in ORDER_BK:
    urut = sorted(ORDER_ST, key=lambda s: -rata(bk, s, "f1_macro"))
    print(f" • {BACKBONES[bk]['nama']}: urutan aktual = {' > '.join(urut)}")
    if {"full", "partial"} <= set(ORDER_ST):
        print(f"   full - partial = {(rata(bk,'full','f1_macro') - rata(bk,'partial','f1_macro')) * 100:+.2f} poin (hipotesis: <= 2 poin)")
    if {"partial", "probing"} <= set(ORDER_ST):
        print(bandingkan(nilai(bk, "partial"), nilai(bk, "probing"), "partial", "probing"))

## 13. Analisis, Rekomendasi, dan Kesimpulan

> ✍️ **Isi bagian ini setelah semua run selesai**, berdasarkan angka di bagian 10–12. Hindari klaim yang tidak didukung data (ingat variasi antar-seed).

### 13.1 Analisis hasil
**a. Pengaruh backbone (Soal 2)**
- *Backbone mana yang memiliki F1 makro tertinggi? Apakah selisihnya lebih besar dari variasi antar-seed?*
- *Bagaimana pengaruh bahasa pretraining (IndoBERT monolingual vs mBERT multibahasa)? Kaitkan dengan statistik tokenisasi di bagian 5.1, misalnya jumlah token per kalimat.*
- *Seberapa besar keuntungan efisiensi DistilmBERT (waktu latih, latensi, parameter, memori) dibanding penurunan performanya?*

**b. Pengaruh strategi fine-tuning (Soal 3)**
- *Seberapa jauh partial freezing dan linear probing tertinggal dari full FT? Apakah polanya konsisten di semua backbone?*
- *Berapa penghematan parameter yang dilatih, waktu latih, dan memori puncak dari strategi pembekuan?*
- *Kelas mana yang paling sulit (lihat F1 per kelas dan confusion matrix)? Biasanya kelas netral, karena jumlahnya paling sedikit.*

**c. Stabilitas antar-seed**
- *Strategi/backbone mana yang paling stabil (std terkecil)? Apakah ada run yang gagal konvergen?*

### 13.2 Rekomendasi (berbasis bukti)
| Skenario | Rekomendasi | Alasan (angka dari tabel) |
|---|---|---|
| Akurasi maksimal, GPU cukup | ... | ... |
| Anggaran komputasi terbatas (latih) | ... | ... |
| Inferensi *real-time* / perangkat terbatas | ... | ... |

### 13.3 Jawaban 3 Pertanyaan Riset Wajib
> **Catatan penting:** teks resmi ketiga pertanyaan riset **tidak termuat** di PDF spesifikasi (dokumen hanya menyebutnya, dan rujukan "lihat bagian C.5" tidak ada di dokumen tersebut). Pertanyaan ini juga tidak ada di modul Pertemuan 9. Mintakan teks resminya ke dosen/LMS, lalu tempel pertanyaan tersebut di bawah sebelum mengisi jawaban.

**PR1.** *(tempel pertanyaan resmi di sini)*
Jawaban: ...

**PR2.** *(tempel pertanyaan resmi di sini)*
Jawaban: ...

**PR3.** *(tempel pertanyaan resmi di sini)*
Jawaban: ...

Panduan mengisi jawaban: rujuk tabel/grafik di bagian 10–12, sebutkan angka beserta simpangan bakunya, dan bedakan mana yang selisihnya lebih besar dari variasi antar-seed.

### 13.4 Kesimpulan
- ...
- ...

### 13.5 Keterbatasan
- Evaluasi memakai data validasi resmi, karena label uji resmi disembunyikan. Hasilnya bisa sedikit berbeda dari angka benchmark IndoNLU di leaderboard.
- Hanya 3 seed dan tanpa pencarian hiperparameter per backbone. Learning rate dikunci per strategi demi keadilan protokol, sehingga belum tentu optimal untuk tiap backbone.
- Selisih waktu latih antar-backbone juga dipengaruhi **jumlah token per kalimat** (ukuran kosakata tiap model berbeda: mBERT/DistilmBERT *cased* vs IndoBERT). Jadi selisih waktu bukan semata-mata efek jumlah lapis, sehingga klaim efisiensi H2 harus dibaca bersama statistik tokenisasi di bagian 5.1.
- Waktu dan latensi bergantung pada GPU dan sesi. Perbandingan yang sah hanya dalam satu sesi (bagian 9).
- Determinisme GPU tidak 100% dijamin. Beberapa operasi CUDA tidak punya versi deterministik, sehingga selisih kecil antar-eksekusi masih mungkin terjadi.

## 14. Kepatuhan Spesifikasi & Berkas yang Dikumpulkan

*Klausul "Ciri-ciri Soal 1/2/3 Sudah Beres" dan "Yang Perlu Dikumpulkan". Sel berikut memeriksa kepatuhan terhadap spesifikasi secara otomatis (khususnya bagian yang baru bisa dinilai setelah ada hasil run), lalu mengemas semua berkas hasil.*

In [ ]:
# ============ GATE STATIS (bagian 2/2) + kepatuhan berbasis HASIL RUN ============
hasil_df = muat_hasil()
print(f"=== KEPATUHAN SPESIFIKASI (mode {MODE}) {('— berkas uji jalan' if MODE == 'uji' else '— berkas resmi')} ===")
if hasil_df.empty:
    print("⚠️  Belum ada hasil run. Jalankan bagian 7 dulu, lalu jalankan ulang sel ini.")
else:
    per_sel = hasil_df.groupby(["backbone", "strategi"])["seed"].nunique()
    cek = {
        "Soal 1: rancangan lengkap & dikonfirmasi": RANCANGAN_DIKONFIRMASI and all(str(v).strip() for v in RANCANGAN.values()),
        "Soal 2: >= 3 backbone diuji": hasil_df["backbone"].nunique() >= 3,
        "Soal 2: BERT, DistilBERT, IndoBERT ada": {"mbert", "distilmbert", "indobert"} <= set(hasil_df["backbone"]),
        "Soal 2: tiap kombinasi >= 3 seed": bool(len(per_sel)) and per_sel.min() >= 3,
        "Soal 2: tabel memuat akurasi, F1, waktu latih, inferensi, parameter": all(
            c in hasil_df for c in ["accuracy", "f1_macro", "train_time_s", "infer_ms_per_kalimat", "params_total"]),
        "Soal 3: full fine-tuning dijalankan": "full" in set(hasil_df["strategi"]),
        "Soal 3: probing/partial dijalankan": bool({"probing", "partial"} & set(hasil_df["strategi"])),
        "Soal 3: >=1 backbone dengan SEMUA strategi": any(
            set(hasil_df[hasil_df["backbone"] == b]["strategi"]) >= set(STRATEGIES) for b in BACKBONES),
        "Soal 3: LR berbeda per strategi": hasil_df.groupby("strategi")["lr"].first().nunique() == hasil_df["strategi"].nunique(),
        "Hal teknis: berkas hasil & rekap dibuat otomatis": HASIL_CSV.exists() and REKAP_CSV.exists(),
        "Hal teknis: inferensi diukur dalam satu sesi": INFER_CSV.exists(),
        "Mode penuh (bukan uji jalan)": MODE == "penuh",
    }
    for k, v in cek.items():
        print(("✅" if v else "❌"), k)
    if MODE == "uji":
        print("ℹ️  Ini berkas uji jalan (UJI_JALAN=True). Untuk laporan resmi: set UJI_JALAN=False, RANCANGAN_DIKONFIRMASI=True, lalu Run all.")

import shutil
arsip = shutil.make_archive(str(OUTPUT_DIR.parent / f"studio3_{DATASET}_{KODE}{SUFIKS}"), "zip",
                            root_dir=OUTPUT_DIR, base_dir=".")
print(f"\n📦 Semua berkas dikemas: {arsip}")
print("Isi folder hasil:")
for f in sorted(OUTPUT_DIR.glob("*")):
    if f.is_file():
        print(f"   - {f.name}")

# ============ Rekap berkas yang wajib dikumpulkan (spesifikasi, bagian "Yang Perlu Dikumpulkan") ============
print("\n=== Yang dikumpulkan ke portal e-learning ===")
print("1. Link notebook Colab/Kaggle (akses: Anyone with the link can view).")
print("2. PDF notebook yang sudah dijalankan penuh (berisi kode, output, dan grafik).")
print("   Nama berkas: STUDIO3_Kelompok15_MuhammadFadhilahAkbar_MuhammadFajriFirdaus.pdf")
print(f"3. Berkas hasil: {HASIL_CSV.name} dan {REKAP_CSV.name} (+ arsip zip bila diperlukan).")
print("4. Slide (maksimal 15) + laporan singkat 3-4 halaman: rancangan, hasil, analisis, jawaban 3 Pertanyaan Riset Wajib.")

In [ ]:
# ====== SIMPAN HASIL KE GOOGLE DRIVE (khusus Google Colab) ======
# Di Colab, OUTPUT_DIR sudah berada di Google Drive sejak bagian 2, jadi setiap run langsung tersimpan.
# Sel ini hanya menyalin bila Drive gagal di-mount di awal (OUTPUT_DIR masih lokal).
import shutil
from pathlib import Path

if DI_DRIVE:
    print(f"✅ Hasil sudah tersimpan langsung di Google Drive: {OUTPUT_DIR}")
    for f in sorted(OUTPUT_DIR.rglob("*")):
        if f.is_file() and f.suffix != ".tsv":
            print("   -", f.relative_to(OUTPUT_DIR))
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/Hasil_Studio3")
        tujuan = DRIVE_DIR / OUTPUT_DIR.name
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        if tujuan.exists() and tujuan.resolve() != OUTPUT_DIR.resolve():
            shutil.rmtree(tujuan)                      # salinan lama diganti yang terbaru
        shutil.copytree(OUTPUT_DIR, tujuan)
        print(f"✅ Hasil disalin ke Google Drive: {tujuan}")
    except ImportError:
        print("Bukan di Colab: langkah Google Drive dilewati (di Kaggle pakai panel Output).")
    except Exception as e:
        print("⚠️ Gagal menyimpan ke Google Drive:", e)


In [ ]:
# ====== UNDUH SEMUA HASIL OTOMATIS (Colab) / ambil dari panel Output (Kaggle) ======
# Sekali Run all, seluruh isi folder hasil dibungkus jadi satu berkas zip lalu diunduh.
import shutil

arsip_semua = shutil.make_archive(str(OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}"), "zip",
                                  root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print(f"📦 Arsip semua hasil: {arsip_semua}")
print("   Isi arsip:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file() and f.suffix != ".tsv":          # TSV data mentah tidak perlu ikut
        print("   -", f.relative_to(OUTPUT_DIR))

try:
    from google.colab import files                     # khusus Google Colab
    for f in [HASIL_CSV, REKAP_CSV, Path(arsip_semua)]:
        files.download(str(f))
    print("\n⬇️  Unduhan dikirim: hasil CSV, rekap agregat, dan arsip zip lengkap.")
except ImportError:
    print(f"\nBukan di Colab: ambil dari panel Output (Kaggle) — berkas ada di {OUTPUT_DIR}")
